# Malaika — Fine-Tune Gemma 4 E4B for Breath Sound Classification

**Pipeline**: Audio WAV → librosa mel-spectrogram → PNG image → Gemma 4 vision

**Dataset**: ICBHI 2017 (920 recordings) | **Method**: PEFT LoRA 4-bit | **Hardware**: Colab T4 GPU

In [ ]:
# Cell 1: Install dependencies
!pip install -q git+https://github.com/huggingface/transformers.git peft trl datasets bitsandbytes accelerate librosa Pillow soundfile kaggle

In [ ]:
# Cell 2: Authenticate (Colab + Kaggle API)
from huggingface_hub import login
import os, subprocess, json, time

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    login(token=secrets.get_secret("HF_TOKEN"))
    KAGGLE_USERNAME = secrets.get_secret("KAGGLE_USERNAME")
    KAGGLE_KEY = secrets.get_secret("KAGGLE_KEY")
    print("Authenticated via Kaggle secrets")
except ModuleNotFoundError:
    try:
        from google.colab import userdata
        login(token=userdata.get("HF_TOKEN"))
        KAGGLE_USERNAME = userdata.get("KAGGLE_USERNAME")
        KAGGLE_KEY = userdata.get("KAGGLE_KEY")
        print("Authenticated via Colab secrets")
    except Exception:
        login()
        KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "")
        KAGGLE_KEY = os.environ.get("KAGGLE_KEY", "")
        print("Authenticated via manual login")

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("Kaggle API configured")

In [ ]:
# Cell 3: Imports + GPU check
import json, os, random, re, subprocess, time
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import librosa
from PIL import Image

print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# Cell 4: Download ICBHI dataset
ICBHI_DIR = Path("/tmp/icbhi_data")
ICBHI_INNER = ICBHI_DIR / "Respiratory_Sound_Database" / "Respiratory_Sound_Database"
KAGGLE_NATIVE = Path("/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database")

if KAGGLE_NATIVE.exists():
    audio_dir = KAGGLE_NATIVE / "audio_and_txt_files"
    print("ICBHI found at Kaggle native path")
elif ICBHI_INNER.exists():
    audio_dir = ICBHI_INNER / "audio_and_txt_files"
    print(f"ICBHI found at {ICBHI_INNER}")
else:
    print("Downloading ICBHI dataset via Kaggle API...")
    ICBHI_DIR.mkdir(parents=True, exist_ok=True)
    dl = subprocess.run(
        ["kaggle", "datasets", "download", "-d", "vbookshelf/respiratory-sound-database",
         "-p", str(ICBHI_DIR), "--unzip"],
        capture_output=True, text=True,
    )
    if dl.returncode == 0:
        print("  Download complete")
        audio_dir = ICBHI_INNER / "audio_and_txt_files"
    else:
        print(f"  Failed: {dl.stderr}")
        audio_dir = None

if audio_dir and audio_dir.exists():
    audio_files = list(audio_dir.glob("*.wav"))
    print(f"ICBHI dataset: {len(audio_files)} audio files")
else:
    print("ICBHI NOT available")
    audio_files = []

In [ ]:
# Cell 5: Parse annotations
def parse_icbhi_annotations(audio_dir):
    records = []
    for txt_file in sorted(audio_dir.glob("*.txt")):
        wav_file = txt_file.with_suffix(".wav")
        if not wav_file.exists():
            continue
        parts = txt_file.stem.split("_")
        patient_id = parts[0] if parts else "unknown"
        has_crackle, has_wheeze, cycle_count = False, False, 0
        with open(txt_file) as f:
            for line in f:
                fields = line.strip().split()
                if len(fields) >= 4:
                    cycle_count += 1
                    if int(fields[2]) == 1: has_crackle = True
                    if int(fields[3]) == 1: has_wheeze = True
        if has_crackle and has_wheeze: label = "both"
        elif has_crackle: label = "crackle"
        elif has_wheeze: label = "wheeze"
        else: label = "normal"
        records.append({"audio_path": str(wav_file), "patient_id": patient_id,
            "label": label, "has_crackle": has_crackle, "has_wheeze": has_wheeze})
    return records

if audio_files:
    records = parse_icbhi_annotations(audio_dir)
    label_counts = Counter(r["label"] for r in records)
    print(f"Parsed {len(records)} recordings:")
    for label, count in sorted(label_counts.items()):
        print(f"  {label}: {count}")

In [ ]:
# Cell 6: Generate spectrograms from audio
SPEC_DIR = Path("/tmp/spectrograms")
SPEC_DIR.mkdir(exist_ok=True)

SPEC_SR, SPEC_N_FFT, SPEC_HOP = 22050, 2048, 512
SPEC_N_MELS, SPEC_FMIN, SPEC_FMAX = 128, 50, 4000
SPEC_WIDTH, SPEC_HEIGHT = 512, 256

def audio_to_spectrogram_image(audio_path, output_path):
    try:
        y, sr = librosa.load(str(audio_path), sr=SPEC_SR, mono=True)
        if len(y) == 0: return False
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=SPEC_N_FFT,
            hop_length=SPEC_HOP, n_mels=SPEC_N_MELS, fmin=SPEC_FMIN, fmax=SPEC_FMAX)
        mel_db = librosa.power_to_db(mel_spec, ref=np.max)
        s_min, s_max = mel_db.min(), mel_db.max()
        norm = ((mel_db - s_min) / (s_max - s_min) * 255).astype(np.uint8) if s_max > s_min else np.zeros_like(mel_db, dtype=np.uint8)
        norm = np.flip(norm, axis=0)
        Image.fromarray(norm, mode="L").resize((SPEC_WIDTH, SPEC_HEIGHT), Image.Resampling.LANCZOS).convert("RGB").save(str(output_path))
        return True
    except Exception as e:
        print(f"  Failed: {Path(audio_path).name}: {e}")
        return False

if audio_files:
    print("Generating spectrograms...")
    spec_records = []
    for i, record in enumerate(records):
        spec_path = SPEC_DIR / f"{Path(record['audio_path']).stem}_spec.png"
        if audio_to_spectrogram_image(record['audio_path'], spec_path):
            record["spectrogram_path"] = str(spec_path)
            spec_records.append(record)
        if (i + 1) % 100 == 0: print(f"  {i+1}/{len(records)} done")
    print(f"\nGenerated {len(spec_records)}/{len(records)} spectrograms")
    records = spec_records

In [ ]:
# Cell 7: Create instruction pairs + train/test split
def _describe(rec):
    if rec["label"] == "normal": return "Normal vesicular breath sounds. No adventitious sounds detected."
    parts = []
    if rec["has_wheeze"]: parts.append("Wheezing detected — continuous horizontal bright bands in 200-1000 Hz")
    if rec["has_crackle"]: parts.append("Crackles detected — discontinuous vertical bright spots")
    return ". ".join(parts) + "."

INSTRUCTION = ("This is a mel-spectrogram image of a child's breathing audio.\n\n"
    "The image shows:\n- Vertical axis: frequency (50 Hz bottom to 4000 Hz top)\n"
    "- Horizontal axis: time (left to right)\n- Brightness: intensity (brighter = louder)\n\n"
    "Classify the breath sounds. Report ONLY a JSON object: "
    '{"wheeze": true/false, "stridor": true/false, "grunting": true/false, '
    '"crackles": true/false, "normal": true/false, '
    '"confidence": <0.0-1.0>, "description": "<what patterns you see>"}')

if audio_files:
    pairs = []
    for r in records:
        pairs.append({"spectrogram_path": r["spectrogram_path"], "label": r["label"],
            "instruction": INSTRUCTION,
            "response": json.dumps({"wheeze": r["has_wheeze"], "stridor": False, "grunting": False,
                "crackles": r["has_crackle"], "normal": r["label"]=="normal",
                "confidence": 0.9, "description": _describe(r)})})
    
    # Patient-level split
    patient_ids = list(set(r["patient_id"] for r in records))
    random.seed(42)
    random.shuffle(patient_ids)
    split = int(len(patient_ids) * 0.8)
    train_pats = set(patient_ids[:split])
    train_pairs = [p for p, r in zip(pairs, records) if r["patient_id"] in train_pats]
    test_pairs = [p for p, r in zip(pairs, records) if r["patient_id"] not in train_pats]
    print(f"Train: {len(train_pairs)} | Test: {len(test_pairs)}")
    for name, sp in [("Train", train_pairs), ("Test", test_pairs)]:
        print(f"  {name}: {dict(Counter(p['label'] for p in sp))}")

In [ ]:
# Cell 8: Load processor + model in 4-bit
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_NAME = "google/gemma-4-E4B-it"
print(f"Loading {MODEL_NAME}...")
t0 = time.monotonic()

processor = AutoProcessor.from_pretrained(MODEL_NAME)
tokenizer = processor.tokenizer

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME, device_map="auto",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4"),
    torch_dtype=torch.float16,
)

print(f"Loaded in {time.monotonic()-t0:.0f}s | VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
# Cell 9: Add LoRA adapter (PEFT)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

# Find target modules (attention + MLP in vision and language)
target_modules = list(set(
    name.split(".")[-1] for name, _ in model.named_modules()
    if any(k in name for k in ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
))
print(f"LoRA target modules: {target_modules}")

model = get_peft_model(model, LoraConfig(
    r=8, lora_alpha=8, lora_dropout=0.0, bias="none",
    target_modules=target_modules, task_type=TaskType.CAUSAL_LM,
))
model.print_trainable_parameters()

In [ ]:
# Cell 10: Build dataset
from torch.utils.data import Dataset as TorchDataset

SYSTEM_MSG = ("You are Malaika, a medical audio analysis assistant following the WHO IMCI protocol. "
    "You analyze mel-spectrogram images of breath sounds to detect abnormalities. "
    "Respond ONLY in the format specified. Do NOT use thinking mode.")

class SpectrogramDataset(TorchDataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        pair = self.pairs[idx]
        spec_img = Image.open(pair["spectrogram_path"]).convert("RGB")
        messages = [
            {"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": f"{SYSTEM_MSG}\n\n{pair['instruction']}"},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": pair["response"]},
            ]},
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        inputs = processor(text=text, images=[spec_img], return_tensors="pt",
                          padding="max_length", truncation=True, max_length=1024)
        input_ids = inputs["input_ids"].squeeze(0)
        attention_mask = inputs["attention_mask"].squeeze(0)
        labels = input_ids.clone()
        # Mask non-response tokens
        resp_len = len(tokenizer.encode(pair["response"], add_special_tokens=False))
        if resp_len > 0 and len(labels) > resp_len:
            labels[:-resp_len] = -100
        result = {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}
        if "pixel_values" in inputs:
            result["pixel_values"] = inputs["pixel_values"].squeeze(0)
        return result

if audio_files:
    train_dataset = SpectrogramDataset(train_pairs)
    print(f"Training dataset: {len(train_dataset)} examples")
else:
    # Dummy data fallback
    SPEC_DIR.mkdir(exist_ok=True)
    dummy_pairs = []
    for i in range(10):
        dp = SPEC_DIR / f"dummy_{i}.png"
        if not dp.exists():
            Image.fromarray(np.random.randint(0, 255, (SPEC_HEIGHT, SPEC_WIDTH, 3), dtype=np.uint8)).save(str(dp))
        dummy_pairs.append({"spectrogram_path": str(dp), "label": "normal",
            "instruction": "Classify breath sounds.",
            "response": '{"wheeze": false, "crackles": false, "normal": true, "confidence": 0.9}'})
    train_dataset = SpectrogramDataset(dummy_pairs)
    print("Using dummy data")

In [ ]:
# Cell 11: Train (100 steps, ~2-3 hours on T4)
from transformers import TrainingArguments, Trainer

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./breath_sound_lora",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        max_steps=100,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=5,
        save_steps=50,
        save_total_limit=2,
        seed=42,
        report_to="none",
        remove_unused_columns=False,
        dataloader_pin_memory=False,
    ),
    train_dataset=train_dataset,
)

print("Starting training...")
result = trainer.train()
print(f"Training complete | Loss: {result.training_loss:.4f} | Steps: {result.global_step}")

In [ ]:
# Cell 12: Save adapter weights
ADAPTER_NAME = "malaika-breath-sounds-spectrogram-lora"

model.save_pretrained(ADAPTER_NAME)
tokenizer.save_pretrained(ADAPTER_NAME)
print(f"Adapter saved to {ADAPTER_NAME}/")

for f in sorted(Path(ADAPTER_NAME).glob("*")):
    if f.is_file():
        print(f"  {f.name}: {f.stat().st_size / 1024 / 1024:.1f} MB")

In [ ]:
# Cell 13: Evaluate on test set
print("=" * 60)
print("EVALUATION ON TEST SET")
print("=" * 60)

if audio_files and test_pairs:
    correct, total_test = 0, 0
    for pair in test_pairs[:20]:
        spec_img = Image.open(pair["spectrogram_path"]).convert("RGB")
        messages = [{"role": "user", "content": [
            {"type": "image"}, {"type": "text", "text": pair["instruction"]}]}]
        input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = processor(text=input_text, images=[spec_img], return_tensors="pt").to(model.device)
        with torch.inference_mode():
            outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        prediction = tokenizer.decode(new_tokens, skip_special_tokens=True)
        try:
            m = re.search(r'\{[^{}]*\}', prediction)
            if m:
                pred = json.loads(m.group(0))
                exp = json.loads(pair["response"])
                if pred.get("wheeze") == exp.get("wheeze") and pred.get("crackles") == exp.get("crackles"):
                    correct += 1; status = "PASS"
                else: status = "FAIL"
            else: status = "FAIL (no JSON)"
        except: status = "FAIL (parse)"
        total_test += 1
        print(f"  {status} [{pair['label']}] -- {prediction[:80]}")
    print(f"\nAccuracy: {correct}/{total_test} ({100*correct/total_test:.0f}%)")
else:
    print("No test data -- add ICBHI dataset to evaluate")

## Summary

| Metric | Value |
|--------|-------|
| Base model | Gemma 4 E4B (4-bit) |
| Method | PEFT LoRA, r=8, vision+language |
| Dataset | ICBHI 2017 (spectrograms) |
| Input | Mel-spectrogram PNG (512x256, 50-4000 Hz) |
| Training steps | 100 |

**Innovation**: Audio → spectrogram → vision fine-tuning bypasses Gemma 4's lack of native audio support.